# 06 · Backtest: knockout step and neutral-venue handling
Questions answered here:
1. Does predicting both team orders and averaging (for neutral venues) beat the single-pass prediction?
2. Does splitting a predicted draw by each team's shootout win rate beat a flat 50/50?

Every model is retrained only on matches **before** the games it is tested on. World Cup knockouts are inferred as the matches after the group stage (first 48 games, or 72 in 2026). Sample sizes: ~71 knockout games, 546 shootouts, ~1,400 neutral-venue games, so small log-loss differences are noise.

Reads `data/processed/{clean_matches,feature_matrix}.csv` and `data/raw/{goalscorers,shootouts}.csv`; writes nothing.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import log_loss
from features import (FEATURE_COLS, build_team_match_index, build_goals_index, build_shootout_index,
                      calculate_elo_history, build_match_features, get_shootout_win_rate)
D='../data'
clean=pd.read_csv(f'{D}/processed/clean_matches.csv',parse_dates=['date'])
goals=pd.read_csv(f'{D}/raw/goalscorers.csv',parse_dates=['date'])
shoot=pd.read_csv(f'{D}/raw/shootouts.csv',parse_dates=['date'])
fm=pd.read_csv(f'{D}/processed/feature_matrix.csv',parse_dates=['date'])
ti=build_team_match_index(clean); gi=build_goals_index(goals); si=build_shootout_index(shoot)
eh,_=calculate_elo_history(clean)
PARAMS=dict(subsample=0.7333333333333333,reg_lambda=0.8947368421052632,reg_alpha=0.15789473684210525,n_estimators=280,
            min_child_weight=8,max_depth=5,learning_rate=0.02351782961998151,colsample_bytree=0.8333333333333333,
            objective='multi:softmax',num_class=3,eval_metric='mlogloss',random_state=42,verbosity=0)
def fit(before):
    tr=fm[fm.date<before]
    m=xgb.XGBClassifier(**PARAMS)
    m.fit(tr[FEATURE_COLS],tr['result'],sample_weight=compute_sample_weight('balanced',tr['result']))
    return m
def feats(h,a,d): return pd.DataFrame([build_match_features(h,a,d,ti,gi,si,eh)])[FEATURE_COLS]
def probs(m,h,a,d): return m.predict_proba(feats(h,a,d))[0]   # [away, draw, home]
def sym(m,h,a,d):
    p=probs(m,h,a,d); q=probs(m,a,h,d)      # q = [h-side as away..]
    return np.array([(p[0]+q[2])/2,(p[1]+q[1])/2,(p[2]+q[0])/2])

## 1. World Cup knockouts (2014, 2018, 2022, 2026 so far)
Target: did the listed home team advance? Methods: **A** flat 50/50 split of the draw, **B** current shootout-rate split (k=3), **C** rate with k=10, **D** order-averaged + 50/50, **E** order-averaged + rate.

In [2]:
wc=clean[clean.tournament=='FIFA World Cup']
ko=[]
for y,ng in [(2014,48),(2018,48),(2022,48),(2026,72)]:
    w=wc[wc.date.dt.year==y].sort_values('date',kind='stable')
    start=w.date.min(); k=w.iloc[ng:]
    m=fit(start)
    for _,r in k.iterrows():
        h,a,d=r.home_team,r.away_team,r.date
        if r.home_score>r.away_score: win=1
        elif r.home_score<r.away_score: win=0
        else:
            s=shoot[(shoot.date==d)&(((shoot.home_team==h)&(shoot.away_team==a))|((shoot.home_team==a)&(shoot.away_team==h)))]
            if len(s)!=1: print('no shootout row',y,h,a,d.date()); continue
            win=int(s.iloc[0].winner==h)
        p=probs(m,h,a,d); ps=sym(m,h,a,d)
        ko.append(dict(year=y,h=h,a=a,date=d,win=win,p=p,ps=ps,
                       sh3=get_shootout_win_rate(h,d,si,k=3),sa3=get_shootout_win_rate(a,d,si,k=3),
                       sh10=get_shootout_win_rate(h,d,si,k=10),sa10=get_shootout_win_rate(a,d,si,k=10)))
K=pd.DataFrame(ko); print('(a 2026 shootout row is missing for Australia v Egypt: shootouts.csv dates it 2026-06-29, the match is 2026-07-03, so that game is skipped)')
print('knockout games:',len(K), K.groupby('year').size().to_dict())
def adv(p,sh,sa): return p[2]+p[1]*sh/(sh+sa)
def ll(y,p): p=np.clip(p,1e-6,1-1e-6); return float(-np.mean(y*np.log(p)+(1-y)*np.log(1-p)))
res={}
res['coin flip (0.5)']=np.full(len(K),0.5)
res['A flat 50/50 split']=np.array([adv(r.p,.5,.5) for r in K.itertuples()])
res['B current (rate k=3)']=np.array([adv(r.p,r.sh3,r.sa3) for r in K.itertuples()])
res['C rate k=10']=np.array([adv(r.p,r.sh10,r.sa10) for r in K.itertuples()])
res['D order-avg + 50/50']=np.array([adv(r.ps,.5,.5) for r in K.itertuples()])
res['E order-avg + rate k=3']=np.array([adv(r.ps,r.sh3,r.sa3) for r in K.itertuples()])
y=K.win.values
print(f"{'method':<28}{'logloss':>9}{'acc':>7}   (n={len(K)}; home team advanced {y.mean():.2f})")
for k,v in res.items(): print(f"{k:<28}{ll(y,v):>9.4f}{(str(round(((v>.5)==y).mean(),2)) if k!='coin flip (0.5)' else '   -'):>7}")
# paired bootstrap vs current
rng=np.random.default_rng(0)
def boot(a,b,n=4000):
    d=[]
    for _ in range(n):
        i=rng.integers(0,len(y),len(y)); d.append(ll(y[i],res[a][i])-ll(y[i],res[b][i]))
    d=np.array(d); return d.mean(), np.percentile(d,2.5), np.percentile(d,97.5)
for a in ['A flat 50/50 split','C rate k=10','D order-avg + 50/50','E order-avg + rate k=3']:
    print(f"logloss({a}) - logloss(current): mean %.4f  95%% CI [%.4f, %.4f]"%boot(a,'B current (rate k=3)'))

no shootout row 2026 Australia Egypt 2026-07-03
(a 2026 shootout row is missing for Australia v Egypt: shootouts.csv dates it 2026-06-29, the match is 2026-07-03, so that game is skipped)
knockout games: 71 {2014: 16, 2018: 16, 2022: 16, 2026: 23}
method                        logloss    acc   (n=71; home team advanced 0.69)
coin flip (0.5)                0.6931      -
A flat 50/50 split             0.6541   0.59
B current (rate k=3)           0.6548   0.59
C rate k=10                    0.6544   0.58
D order-avg + 50/50            0.6200    0.7
E order-avg + rate k=3         0.6206   0.62


logloss(A flat 50/50 split) - logloss(current): mean -0.0006  95% CI [-0.0158, 0.0143]
logloss(C rate k=10) - logloss(current): mean -0.0003  95% CI [-0.0069, 0.0061]
logloss(D order-avg + 50/50) - logloss(current): mean -0.0344  95% CI [-0.0663, -0.0029]
logloss(E order-avg + rate k=3) - logloss(current): mean -0.0340  95% CI [-0.0621, -0.0067]


## 2. Shootout win rate vs a coin flip, all shootouts since 1990

In [3]:
rows=[]
for _,r in shoot.iterrows():
    if r.date<pd.Timestamp('1990-01-01'): continue
    rows.append((int(r.winner==r.home_team),
        get_shootout_win_rate(r.home_team,r.date,si,k=3),get_shootout_win_rate(r.away_team,r.date,si,k=3),
        get_shootout_win_rate(r.home_team,r.date,si,k=10),get_shootout_win_rate(r.away_team,r.date,si,k=10)))
R=np.array(rows); yy=R[:,0]
print(f"\nshootouts since 1990: {len(yy)}; first-listed team won {yy.mean():.3f}")
for name,p in [('coin flip',np.full(len(yy),.5)),('rate k=3',R[:,1]/(R[:,1]+R[:,2])),('rate k=10',R[:,3]/(R[:,3]+R[:,4]))]:
    nt=(p!=.5); acc=f"{((p>.5)==yy)[nt].mean():.3f} (on {nt.sum()} games where the rates differ)" if nt.any() else "n/a (always 50/50)"
    print(f"  {name:<10} logloss {ll(yy,p):.4f}  acc {acc}")


shootouts since 1990: 546; first-listed team won 0.548


  coin flip  logloss 0.6931  acc n/a (always 50/50)
  rate k=3   logloss 0.7093  acc 0.498 (on 482 games where the rates differ)
  rate k=10  logloss 0.6979  acc 0.492 (on 482 games where the rates differ)


## 3. Neutral-venue matches 2018-2026: single pass vs order-averaged
Model retrained on every 1 January using only earlier matches.

In [4]:
out=[]
for Y in range(2018,2027):
    m=fit(pd.Timestamp(f'{Y}-01-01'))
    t=clean[(clean.date.dt.year==Y)&(clean.neutral==True)]
    for _,r in t.iterrows():
        res3=2 if r.home_score>r.away_score else (1 if r.home_score==r.away_score else 0)
        out.append((Y,res3,probs(m,r.home_team,r.away_team,r.date),sym(m,r.home_team,r.away_team,r.date)))
yt=np.array([o[1] for o in out]); P=np.array([o[2] for o in out]); PS=np.array([o[3] for o in out])
print(f"\nneutral-venue matches 2018-2026 (model retrained each Jan 1): n={len(yt)}")
print(f"  original    logloss {log_loss(yt,P,labels=[0,1,2]):.4f}  acc {(P.argmax(1)==yt).mean():.3f}")
print(f"  order-avg   logloss {log_loss(yt,PS,labels=[0,1,2]):.4f}  acc {(PS.argmax(1)==yt).mean():.3f}")
print(f"  mean P(home win): original {P[:,2].mean():.3f}  order-avg {PS[:,2].mean():.3f}   actual {(yt==2).mean():.3f}")


neutral-venue matches 2018-2026 (model retrained each Jan 1): n=1421
  original    logloss 0.9444  acc 0.536
  order-avg   logloss 0.9319  acc 0.552
  mean P(home win): original 0.342  order-avg 0.326   actual 0.393
